In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TranslationPipeline
from transformers import pipeline

In [3]:
text = ['과장님 안녕하십니까.', '오늘 처음 들어온 신입입니다.','아 개힘드네','제발 좀 그만해']

ko_zero_shot_clf = pipeline('zero-shot-classification',model='MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli')

labels = ["무례하고 공격적임", "예의 바르고 정중함"]
shot_word = ko_zero_shot_clf(text,labels,hypothesis_template="이 문장은 직장 상사에게 {} 느낌을 준다.")
shot_word

Device set to use cpu


[{'sequence': '과장님 안녕하십니까.',
  'labels': ['예의 바르고 정중함', '무례하고 공격적임'],
  'scores': [0.5612726211547852, 0.4387274384498596]},
 {'sequence': '오늘 처음 들어온 신입입니다.',
  'labels': ['예의 바르고 정중함', '무례하고 공격적임'],
  'scores': [0.6655117273330688, 0.3344883322715759]},
 {'sequence': '아 개힘드네',
  'labels': ['무례하고 공격적임', '예의 바르고 정중함'],
  'scores': [0.6947562098503113, 0.3052437901496887]},
 {'sequence': '제발 좀 그만해',
  'labels': ['무례하고 공격적임', '예의 바르고 정중함'],
  'scores': [0.6802514791488647, 0.31974852085113525]}]

In [4]:
from konlpy.tag import Okt

okt = Okt()
word_list = []
for k in range(len(shot_word)):
    
    if shot_word[k]['labels'][0] == '무례하고 공격적임':
        target_sentence = shot_word[k]['sequence']

        result = okt.pos(target_sentence)

        i = 0
        
        while i < len(result):
            word, pos = result[i]

            if pos == 'Modifier':
                word1, pos1 = result[i+1]
                word = word + word1 
                i += 2
                word_list.append(word)
            else:
                i += 1

            if pos in ['Noun','Verb','Adjective','Exclamation']:
                word_list.append(word)
word_list


['아', '개', '힘드네', '제발', '좀', '그만해']

In [ ]:
labels = ["공격적인 비속어", "무례한 반말", "짜증과 불평", "일반적인 단어"]

target_list = []   

for word in word_list:
    result = ko_zero_shot_clf(word, labels, 
                           hypothesis_template=f"'{target_sentence}'라는 문장에서 '{word}'는 {{}} 느낌을 준다.")
    
    top_label = result['labels'][0]
    print(f'단어 : {word} | 태도: {top_label}')

    if top_label in ["공격적인 비속어", "무례한 반말", "짜증과 불평"]:
        target_list.append(word)

target_list


단어 : 아 | 태도: 무례한 반말
단어 : 개 | 태도: 무례한 반말
단어 : 힘드네 | 태도: 무례한 반말
단어 : 제발 | 태도: 무례한 반말
단어 : 좀 | 태도: 짜증과 불평
단어 : 그만해 | 태도: 무례한 반말


['아', '개', '힘드네', '제발', '좀', '그만해']